In [ ]:
import pandas as pd
import matplotlib.pyplot as plt 
import os
import numpy as np
from scipy.stats import sem
import scipy
from cycler import cycler
%matplotlib inline

In [ ]:
dataset_name = ["DTD", "EuroSAT", "GTSRB", "MNIST", "RESISC45", "Stanford_Cars", "SUN397", "SVHN"] # DTD | EuroSAT | GTSRB | MNIST | RESISC45 | Stanford_Cars | SUN397 | SVHN
refining_type = "Standard" # "Standard" | "Increment_Training"
refiner = "Fine_Tuned" # "Fine_Tuned" | "Reverse_Probe"
model_name = "CLIP_ViT_Vision" # CLIP_ViT_Vision | DeiT 
domain = "Base_Fine_Tuned" # "Base_Fine_Tuned" | "Fine_Tuned_Layer_Skipping"
transformation = ["Standard", "Base_Fine_Tuned_Classifier", "Base_Linear_Probe"] # "Standard" | "Base_Fine_Tuned_Classifier" | "Base_Linear_Probe"
indices = [i for i in range(12)]
reps = [i for i in range(1,6)]

## Core Graphs

In [ ]:
for k in range(0, len(dataset_name)): # len(dataset_name)
    results_path = f"./{model_name}/Data/{refining_type}/{refiner}/{dataset_name[k]}/{domain}" # /Entire_Transformation_Matrix_W"

    # 1-5
    results = {}

    for rep in reps:
        results[rep] = []
        path = f"{results_path}/{rep}/Entire_Transformation_Matrix_W"
        try:
            for filename in os.listdir(path):
                if filename in [".DS_Store", f"Base_Fine_Tuned_Classifier_Results_{rep}.json", f"Base_Linear_Probe_Results_{rep}.json", ".ipynb_checkpoints"]: 
                    continue
                file_path = os.path.join(path, filename)
                if os.path.isfile(file_path):
                    results[rep].append(file_path)
        except FileNotFoundError:
            print(f"Error: The Folder '{path}' was not found.")
        except Exception as e:
            print(f"An error occured: {e}")

        data = []
        for filepath in results[rep]:
            try:
                df = pd.read_json(filepath)
                data.append(df)
            except ValueError as ve:
                print(f"Failed to read JSON from file: {filepath} | Error: {ve}")
            except Exception as e:
                print(f"Other error with file {filepath}: {e}")
        data = sorted(data, key=lambda df: df["Train_Data_Size"][0])
        
        results[rep] = data

In [ ]:
mean_results = {}
top_error_bar = {}
bottom_error_bar = {}

for indice in indices:
    mean_results[indice] = []
    top_error_bar[indice] = []
    bottom_error_bar[indice] = []
    for file in range(len(results[1])):
        temp_acc = []
        for rep in range(1, len(results)+1):
            temp_acc.append(results[rep][file]["Classification_Accuracy"][indice])
        ci_low, ci_high = scipy.stats.t.interval(0.95, df=len(temp_acc)-1, loc=np.mean(temp_acc), scale=sem(temp_acc))
        bottom_error_bar[indice].append(ci_low)
        top_error_bar[indice].append(ci_high)
        mean_results[indice].append(np.mean(temp_acc))

In [ ]:
fine_tune_mean = 0
fine_tune_top_error = 0
fine_tune_bottom_error = 0
for k in range(6, 7):
    refined_acc = []
    df = pd.read_json(f"./{model_name}/Data/{refining_type}/Refined_Accuracy/Fine_Tuned/{dataset_name[k]}_Accuracy.json")

    for i in range(5):
        refined_acc.append(df["Accuracy"][i])
    fine_tune_mean = np.mean(refined_acc)

    fine_tune_bottom_error, fine_tune_top_error = scipy.stats.t.interval(0.95, df=len(refined_acc)-1, loc=fine_tune_mean, scale=sem(refined_acc))

In [ ]:
linear_probe_mean = 0
linear_probe_top_error = 0
linear_probe_bottom_error = 0
for k in range(6, 7):
    refined_acc = []
    df = pd.read_json(f"./{model_name}/Data/{refining_type}/Refined_Accuracy/Linear_Probe/{dataset_name[k]}_Accuracy.json")

    for i in range(5):
        refined_acc.append(df["Accuracy"][i])
    linear_probe_mean = np.mean(refined_acc)

    linear_probe_bottom_error, linear_probe_top_error = scipy.stats.t.interval(0.95, df=len(refined_acc)-1, loc=fine_tune_mean, scale=sem(refined_acc))

In [ ]:
train_size = []
for i in range(len(results[1])):
    train_size.append(results[1][i]["Train_Data_Size"][0][0])

my_colors = [
    '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
        "#528c4b", '#e377c2', '#7f7f7f', "#97b47c", '#17becf',
        '#a6cee3', '#b2df8a', '#fb9a99', "#c77b18", "#658a92"
]

plt.rcParams['axes.prop_cycle'] = cycler(color=my_colors)

plt.plot([0, train_size[-1]], [fine_tune_mean, fine_tune_mean], label="Fine Tuned", linestyle="--")
plt.plot([0, train_size[-1]], [linear_probe_mean, linear_probe_mean], label="Linear Probe", linestyle="--")
plt.plot([0, train_size[-1]], [1/397, 1/397], label="Base (Random)", linestyle='--')

for i in indices:
    plt.plot(train_size, mean_results[i], marker="o", label=f"Layer {i}")


plt.xlabel("Number of Training Images")
plt.ylabel("Classification Accuracy (%)")
plt.legend(
    loc="center left",
    bbox_to_anchor=(1.0, 0.5)
)
plt.title(f"Task Matrices: CLIP ViT B/32 Vision - SUN397")
plt.savefig("./CLIP_ViT_Vision_SUN397", dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
# Dataset groups
text_datasets = ['BLiMP', 'HANS', 'TREC-6']
vision_datasets = ['SUN397', 'GTSRB', 'MNIST']

# Data (aligned by dataset)
base =        [15.3, 63.4, 42.5, 65.3, 45.5, 48.9]
linear_probe =[38.1, 76, 75.1, 73.8, 86.8, 98.7]
task_matrix = [50, 82.3, 84.7, 74.8, 87.2, 99.03]
fine_tuned =  [60.5, 99.4, 93.2, 74.5, 98.7, 99.4]
baseline_random = [1/67 * 100, 1/2 * 100, 1/6 * 100, 1/397 * 100, 1/43 * 100, 1/10 * 100]

# Setup
fig, axs = plt.subplots(1, 2, figsize=(9, 7), sharey=True)

bar_width = 0.2

def plot_group(ax, indices, labels, remove_spine):
    x = np.arange(len(indices)) * 0.3

    colors = {
        'Base': 'skyblue',
        'Linear Probe': '#1f78b4',
        'Task Matrix': '#08306b',
        'Fine-Tuned': '#66c2a5'
    }

    for i, idx in enumerate(indices):
        # Values capped at 100
        vals = {
            'Base': min(base[idx], 100),
            'Linear Probe': min(linear_probe[idx], 100),
            'Task Matrix': min(task_matrix[idx], 100),
            'Fine-Tuned': min(fine_tuned[idx], 100)
        }

        # Draw bars in the specific order (back to front)
        order = ['Fine-Tuned', 'Task Matrix', 'Linear Probe', 'Base']

        for label in order:
            ax.bar(x[i], vals[label], width=bar_width, color=colors[label],
                   label=label if (i == 0) else "", alpha=1.0, zorder=order.index(label)+1)

        # Baseline line on top
        left = x[i] - bar_width / 2
        right = x[i] + bar_width / 2
        ax.hlines(y=baseline_random[idx], xmin=left, xmax=right, colors='red', linestyles='dashed',
                  label='Baseline (Random)' if i == 0 else "", linewidth=1.5, zorder=10)

    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=14)
    ax.set_ylim(0, 105)
    ax.grid(axis='y', linestyle='--', alpha=0.6)

    # Remove specific spines
    if remove_spine == 'right':
        ax.spines['right'].set_visible(False)
    elif remove_spine == 'left':
        ax.spines['left'].set_visible(False)

    for spine in ax.spines.values():
        spine.set_color('lightgrey')

    ax.tick_params(axis='both', which='both', length=0)

# Plot text datasets (left subplot)
plot_group(axs[0], [0, 1, 2], text_datasets, remove_spine='right')
axs[0].set_title("allMiniLM-L12-V2", fontsize=12)

# Plot vision datasets (right subplot)
plot_group(axs[1], [3, 4, 5], vision_datasets, remove_spine='left')
axs[1].set_title("CLIP ViT-B/32 Vision", fontsize=12)

# Shared Y label (grey)
fig.text(0.02, 0.5, 'Accuracy (%)', va='center', rotation='vertical', fontsize=13, color='grey')

# Legend (bottom)
handles, labels = axs[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.5, -0.05), ncol=5, frameon=True, fontsize=12)

plt.tight_layout(rect=[0.03, 0, 1, 1])
plt.savefig("./addp_results", dpi=600, bbox_inches='tight')
plt.show()

## Multi-Class Augmentation

In [ ]:
import itertools
import json

all_combinations = {}

for r in range(1, 9):  # lengths 2 through 8
    combos = list(itertools.combinations(dataset_name, r))
    all_combinations[r] = combos

### Single Graphs

In [ ]:
multi_class_scores = {}
multi_class_scores_error_top = {}
multi_class_scores_error_bottom = {}

for set in range(1,9):
    multi_class_scores[set] = {}
    multi_class_scores_error_top[set] = {}
    multi_class_scores_error_bottom[set] = {}
    for combo in range(len(all_combinations[set])):
        results_path = f"./{model_name}/Data/Multi_Class_Augmentation/{refining_type}/{refiner}/Performance"
        temp_acc = {}
        matrix_name = ""
        for c in all_combinations[set][combo]:
            matrix_name += f"{c}_"
            temp_acc[c] = {i: [] for i in indices}
        
        for rep in range(1,6):
            try: 
                with open(f"{results_path}/{set}/{domain}/{matrix_name}/{rep}.json", "r") as file:
                    data = json.load(file)
            except FileNotFoundError:
                print("Error: data.json not found.")
            for c in all_combinations[set][combo]:
                for i in indices:
                    temp_acc[c][i].append(data["Classification_Accuracy"][c][str(i)])
        # temp_acc (dataset_name, layer, accuracy)

        multi_class_scores[set][matrix_name] = {}
        multi_class_scores_error_top[set][matrix_name] = {}
        multi_class_scores_error_bottom[set][matrix_name] = {}

        for c in all_combinations[set][combo]:
            multi_class_scores[set][matrix_name][c] = {}
            multi_class_scores_error_top[set][matrix_name][c] = {}
            multi_class_scores_error_bottom[set][matrix_name][c] = {}
            for i in indices:
                fine_tune_mean = np.mean(temp_acc[c][i])
                multi_class_scores[set][matrix_name][c][i] = fine_tune_mean
                b, t = scipy.stats.t.interval(0.95, df=len(temp_acc[c][i])-1, loc=fine_tune_mean, scale=sem(temp_acc[c][i]))
                multi_class_scores_error_top[set][matrix_name][c][i] = b
                multi_class_scores_error_bottom[set][matrix_name][c][i] = t

In [ ]:
fine_tuned_accuracy = {}
for k in range(0, len(dataset_name)):
    refined_acc = []
    df = pd.read_json(f"./{model_name}/Data/Core/{refining_type}/Refined_Accuracy/Fine_Tuned/{dataset_name[k]}_Accuracy.json")

    for i in range(5):
        refined_acc.append(df["Accuracy"][i])

    print(f"{model_name} - {dataset_name[k]}: Fine Tuned Accuracy")
    ci_low, ci_high = scipy.stats.t.interval(0.95, df=len(refined_acc)-1, loc=np.mean(refined_acc), scale=sem(refined_acc))
    print(f"Fine-Tuned Average: {np.mean(refined_acc)}, Error: +- {ci_high-np.mean(refined_acc)}, 95% Confidence Interval: {(ci_low, ci_high)}\n")
    fine_tuned_accuracy[dataset_name[k]] = np.mean(refined_acc)

In [ ]:
marker_list = []
x_acc = []
y_acc = []
labels = []
box_name = []
color_name = []
marker_list = ["^", "H", "X", "D", "s", "*", "+", "p"]
color_list = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple', 'tab:brown', 'tab:pink', 'tab:gray', 'tab:olive', 'tab:cyan']
color_counter = 0
prev = None

for combo in range(len(all_combinations[2])):
    first = all_combinations[2][combo][0]
    matrix_name = ""
    for k in range(0, len(all_combinations[2][combo])):
        matrix_name += f"{all_combinations[2][combo][k]}_"
    label_name = f"{all_combinations[2][combo][0]}"
    for k in range(1, len(all_combinations[2][combo])):
        label_name += f", {all_combinations[2][combo][k]}"
    labels.append(label_name)
    box_name.append(marker_list[dataset_name.index(all_combinations[2][combo][0])])

    if first != prev:
        color_counter = 0
    
    prev = first
    color_name.append(color_list[color_counter])
    color_counter += 1

    x_acc.append(multi_class_scores[2][matrix_name][all_combinations[2][combo][0]][11] / fine_tuned_accuracy[all_combinations[2][combo][0]])
    y_acc.append(multi_class_scores[2][matrix_name][all_combinations[2][combo][1]][11] / fine_tuned_accuracy[all_combinations[2][combo][1]])

In [ ]:
for x, y, label, marker, color_n in zip(x_acc, y_acc, labels, box_name, color_name):
    plt.scatter(x, y, label=label, marker=marker, color=color_n)

plt.axhline(y=1, color="grey", linestyle='--', linewidth=2, label='Fine-Tuned Normalized Accuracy')
plt.axvline(x=1, color="grey", linestyle='--', linewidth=2)


plt.legend(
    loc='center left',
    bbox_to_anchor=(2.0, 0.5),
    ncol=2,
    fontsize='small'
)
plt.xlabel("Normalized Task 1 Accuracy")
plt.ylabel("Normalized Task 2 Accuracy")
plt.title("CLIP ViT Vision Multi-Task (2) Task Matrices")
plt.savefig("Multi_Class_Augmentation_2_Results.png", dpi=600, bbox_inches='tight')
plt.show()

### Double Graphs

In [ ]:
refining_type = "Standard" # "Standard" | "Increment_Training"
refiner = "Fine_Tuned" # "Fine_Tuned" | "Reverse_Probe"

multi_class_scores_1 = {}
multi_class_scores_error_top_1 = {}
multi_class_scores_error_bottom_1 = {}

for set in range(1,9):
    multi_class_scores_1[set] = {}
    multi_class_scores_error_top_1[set] = {}
    multi_class_scores_error_bottom_1[set] = {}
    for combo in range(len(all_combinations[set])):
        results_path = f"./{model_name}/Data/Multi_Class_Augmentation/{refining_type}/{refiner}/Performance"
        temp_acc = {}
        matrix_name = ""
        for c in all_combinations[set][combo]:
            matrix_name += f"{c}_"
            temp_acc[c] = {i: [] for i in indices}
        
        for rep in range(1,6):
            try: 
                with open(f"{results_path}/{set}/{domain}/{matrix_name}/{rep}.json", "r") as file:
                    data = json.load(file)
            except FileNotFoundError:
                print("Error: data.json not found.")
            for c in all_combinations[set][combo]:
                for i in indices:
                    temp_acc[c][i].append(data["Classification_Accuracy"][c][str(i)])
        # temp_acc (dataset_name, layer, accuracy)

        multi_class_scores_1[set][matrix_name] = {}
        multi_class_scores_error_top_1[set][matrix_name] = {}
        multi_class_scores_error_bottom_1[set][matrix_name] = {}

        for c in all_combinations[set][combo]:
            multi_class_scores_1[set][matrix_name][c] = {}
            multi_class_scores_error_top_1[set][matrix_name][c] = {}
            multi_class_scores_error_bottom_1[set][matrix_name][c] = {}
            for i in indices:
                fine_tune_mean = np.mean(temp_acc[c][i])
                multi_class_scores_1[set][matrix_name][c][i] = fine_tune_mean
                b, t = scipy.stats.t.interval(0.95, df=len(temp_acc[c][i])-1, loc=fine_tune_mean, scale=sem(temp_acc[c][i]))
                multi_class_scores_error_top_1[set][matrix_name][c][i] = b
                multi_class_scores_error_bottom_1[set][matrix_name][c][i] = t

In [ ]:
fine_tuned_accuracy_1 = {}
for k in range(0, len(dataset_name)):
    refined_acc = []
    df = pd.read_json(f"./{model_name}/Data/Core/{refining_type}/Refined_Accuracy/Fine_Tuned/{dataset_name[k]}_Accuracy.json")

    for i in range(5):
        refined_acc.append(df["Accuracy"][i])

    print(f"{model_name} - {dataset_name[k]}: Fine Tuned Accuracy")
    ci_low, ci_high = scipy.stats.t.interval(0.95, df=len(refined_acc)-1, loc=np.mean(refined_acc), scale=sem(refined_acc))
    print(f"Fine-Tuned Average: {np.mean(refined_acc)}, Error: +- {ci_high-np.mean(refined_acc)}, 95% Confidence Interval: {(ci_low, ci_high)}\n")
    fine_tuned_accuracy_1[dataset_name[k]] = np.mean(refined_acc)

In [ ]:
refining_type = "Increment_Training" # "Standard" | "Increment_Training"
refiner = "Fine_Tuned" # "Fine_Tuned" | "Reverse_Probe"

multi_class_scores_2 = {}
multi_class_scores_error_top_2 = {}
multi_class_scores_error_bottom_2 = {}

for set in range(1,9):
    multi_class_scores_2[set] = {}
    multi_class_scores_error_top_2[set] = {}
    multi_class_scores_error_bottom_2[set] = {}
    for combo in range(len(all_combinations[set])):
        results_path = f"./{model_name}/Data/Multi_Class_Augmentation/{refining_type}/{refiner}/Performance"
        temp_acc = {}
        matrix_name = ""
        for c in all_combinations[set][combo]:
            matrix_name += f"{c}_"
            temp_acc[c] = {i: [] for i in indices}
        
        for rep in range(1,6):
            try: 
                with open(f"{results_path}/{set}/{domain}/{matrix_name}/{rep}.json", "r") as file:
                    data = json.load(file)
            except FileNotFoundError:
                print("Error: data.json not found.")
            for c in all_combinations[set][combo]:
                for i in indices:
                    temp_acc[c][i].append(data["Classification_Accuracy"][c][str(i)])
        # temp_acc (dataset_name, layer, accuracy)

        multi_class_scores_2[set][matrix_name] = {}
        multi_class_scores_error_top_2[set][matrix_name] = {}
        multi_class_scores_error_bottom_2[set][matrix_name] = {}

        for c in all_combinations[set][combo]:
            multi_class_scores_2[set][matrix_name][c] = {}
            multi_class_scores_error_top_2[set][matrix_name][c] = {}
            multi_class_scores_error_bottom_2[set][matrix_name][c] = {}
            for i in indices:
                fine_tune_mean = np.mean(temp_acc[c][i])
                multi_class_scores_2[set][matrix_name][c][i] = fine_tune_mean
                b, t = scipy.stats.t.interval(0.95, df=len(temp_acc[c][i])-1, loc=fine_tune_mean, scale=sem(temp_acc[c][i]))
                multi_class_scores_error_top_2[set][matrix_name][c][i] = b
                multi_class_scores_error_bottom_2[set][matrix_name][c][i] = t

In [ ]:
fine_tuned_accuracy_2 = {}
for k in range(0, len(dataset_name)):
    refined_acc = []
    df = pd.read_json(f"./{model_name}/Data/Core/{refining_type}/Refined_Accuracy/Fine_Tuned/{dataset_name[k]}_Accuracy.json")

    for i in range(5):
        refined_acc.append(df["Accuracy"][i])

    print(f"{model_name} - {dataset_name[k]}: Fine Tuned Accuracy")
    ci_low, ci_high = scipy.stats.t.interval(0.95, df=len(refined_acc)-1, loc=np.mean(refined_acc), scale=sem(refined_acc))
    print(f"Fine-Tuned Average: {np.mean(refined_acc)}, Error: +- {ci_high-np.mean(refined_acc)}, 95% Confidence Interval: {(ci_low, ci_high)}\n")
    fine_tuned_accuracy_2[dataset_name[k]] = np.mean(refined_acc)

In [ ]:
x_acc_1 = []
y_acc_1 = []
labels_1 = []
box_name_1 = []
color_name_1 = []
marker_list = ["^", "H", "X", "D", "s", "*", "+", "p"]
color_list = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple', 'tab:brown', 'tab:pink', 'tab:gray', 'tab:olive', 'tab:cyan']
color_counter = 0
prev = None

for combo in range(len(all_combinations[2])):
    first = all_combinations[2][combo][0]
    matrix_name = ""
    for k in range(0, len(all_combinations[2][combo])):
        matrix_name += f"{all_combinations[2][combo][k]}_"
    label_name = f"{all_combinations[2][combo][0]}"
    for k in range(1, len(all_combinations[2][combo])):
        label_name += f", {all_combinations[2][combo][k]}"
    labels_1.append(label_name)
    box_name_1.append(marker_list[dataset_name.index(all_combinations[2][combo][0])])

    if first != prev:
        color_counter = 0
    
    prev = first
    color_name_1.append(color_list[color_counter])
    color_counter += 1

    x_acc_1.append(multi_class_scores_1[2][matrix_name][all_combinations[2][combo][0]][11] / fine_tuned_accuracy_1[all_combinations[2][combo][0]])
    y_acc_1.append(multi_class_scores_1[2][matrix_name][all_combinations[2][combo][1]][11] / fine_tuned_accuracy_1[all_combinations[2][combo][1]])

In [ ]:
x_acc_2 = []
y_acc_2 = []
labels_2 = []
box_name_2 = []
color_name_2 = []
marker_list = ["^", "H", "X", "D", "s", "*", "+", "p"]
color_list = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple', 'tab:brown', 'tab:pink', 'tab:gray', 'tab:olive', 'tab:cyan']
color_counter = 0
prev = None

for combo in range(len(all_combinations[2])):
    first = all_combinations[2][combo][0]
    matrix_name = ""
    for k in range(0, len(all_combinations[2][combo])):
        matrix_name += f"{all_combinations[2][combo][k]}_"
    label_name = f"{all_combinations[2][combo][0]}"
    for k in range(1, len(all_combinations[2][combo])):
        label_name += f", {all_combinations[2][combo][k]}"
    labels_2.append(label_name)
    box_name_2.append(marker_list[dataset_name.index(all_combinations[2][combo][0])])

    if first != prev:
        color_counter = 0
    
    prev = first
    color_name_2.append(color_list[color_counter])
    color_counter += 1

    x_acc_2.append(multi_class_scores_2[2][matrix_name][all_combinations[2][combo][0]][11] / fine_tuned_accuracy_2[all_combinations[2][combo][0]])
    y_acc_2.append(multi_class_scores_2[2][matrix_name][all_combinations[2][combo][1]][11] / fine_tuned_accuracy_2[all_combinations[2][combo][1]])

In [ ]:
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(30, 6))
gs = gridspec.GridSpec(1, 3, width_ratios=[1, 0.4, 1], wspace=0.3)

ax1 = fig.add_subplot(gs[0])
legend_ax = fig.add_subplot(gs[1])
ax2 = fig.add_subplot(gs[2])

for x, y, label, marker, color_n in zip(x_acc_1, y_acc_1, labels_1, box_name_1, color_name_1):
    ax1.scatter(x, y, label=label, marker=marker, color=color_n)

ax1.axhline(y=1, color="grey", linestyle='--', linewidth=2, label='Fine-Tuned Normalized Accuracy')
ax1.axvline(x=1, color="grey", linestyle='--', linewidth=2)

ax1.set_xlabel("Normalized Task 1 Accuracy")
ax1.set_ylabel("Normalized Task 2 Accuracy")
ax1.set_title("CLIP ViT Vision Full Train Multi-Task (2) Task Matrices")

for x, y, label, marker, color_n in zip(x_acc_2, y_acc_2, labels_2, box_name_2, color_name_2):
    ax2.scatter(x, y, label=label, marker=marker, color=color_n)

ax2.axhline(y=1, color="grey", linestyle='--', linewidth=2, label='Fine-Tuned Normalized Accuracy')
ax2.axvline(x=1, color="grey", linestyle='--', linewidth=2)

ax2.set_xlabel("Normalized Task 1 Accuracy")
ax2.set_ylabel("Normalized Task 2 Accuracy")
ax2.set_title("CLIP ViT Vision 20% Train Multi-Task (2) Task Matrices")

handles, labels = ax1.get_legend_handles_labels()

# Turn off legend axis and place legend there
legend_ax.axis('off')
legend_ax.legend(handles, labels, loc='center', ncol=2, frameon=True)

plt.savefig("Multi_Class_Augmentation_2_Results.png", dpi=600, bbox_inches='tight')
plt.show()

## All Train Images Layerwise Graphs

### Collecting Data

In [ ]:
def find_best_acc(data):
    acc = {i: [] for i in indices} # Indices, then len(data)
    for i in indices:
        for j in range(len(data)):
            acc[i].append(data[j]["Classification_Accuracy"][i])

    best_acc = {}

    for i in acc:
        arr = acc[i]
        max_val = max(arr)
        indice = arr.index(max_val)
        num_img = data[indice]["Train_Data_Size"][i][0]
        best_acc[i] = (max_val, num_img)

    final = []
    num_img = []
    for i in indices:
        final.append(best_acc[i][0])
        num_img.append(best_acc[i][1])

    best_accuracy = max(final)
    index = final.index(best_accuracy)
    best_accuracy_num_images = num_img[index]

    return best_accuracy, best_accuracy_num_images, index

In [ ]:
def retrieve_val(path):
    best_data = pd.read_json(path)
    acc = {}
    for i in indices:
        acc[i] = best_data["Classification_Accuracy"][i]
    return acc

In [ ]:
# Task Matrix

best_layerwise_acc = {}
best_layerwise_img = {}
best_layerwise_index = {}

for ds_indice in range(len(dataset_name)):
    results_path = f"./{model_name}/Data/Core/{refining_type}/{refiner}/{dataset_name[ds_indice]}/{domain}"
    results = {}
    best_layerwise_acc[dataset_name[ds_indice]] = {}
    best_layerwise_img[dataset_name[ds_indice]] = {}
    best_layerwise_index[dataset_name[ds_indice]] = {}

    for rep in range(1,6):
        results[rep] = []
        path = f"{results_path}/{rep}/Entire_Transformation_Matrix_W"
        try:
            for filename in os.listdir(path):
                if filename in [".DS_Store", f"Base_Fine_Tuned_Classifier_Results_{rep}.json", f"Base_Linear_Probe_Results_{rep}.json", ".ipynb_checkpoints"]: 
                    continue
                file_path = os.path.join(path, filename)
                if os.path.isfile(file_path):
                    results[rep].append(file_path)
        except FileNotFoundError:
            print(f"Error: The Folder '{path}' was not found.")
        except Exception as e:
            print(f"An error occured: {e}")

        data = []
        for filepath in results[rep]:
            try:
                df = pd.read_json(filepath)
                data.append(df)
            except ValueError as ve:
                print(f"Failed to read JSON from file: {filepath} | Error: {ve}")
            except Exception as e:
                print(f"Other error with file {filepath}: {e}")
        results[rep] = data

    best_acc = []
    num_img = []
    index = []
    
    for rep in range(1,6):
        acc, img, ind = find_best_acc(results[rep])
        best_acc.append(acc)
        num_img.append(img)
        index.append(ind)
        best_layerwise_acc[dataset_name[ds_indice]][rep] = retrieve_val(f"{results_path}/{rep}/Entire_Transformation_Matrix_W/Standard_{img}_Results_{rep}.json")
        best_layerwise_img[dataset_name[ds_indice]][rep] = img
        best_layerwise_index[dataset_name[ds_indice]][rep] = index

In [ ]:
for ds_indice in range(len(dataset_name)):
    for rep in range(1,6):
        print(f"{dataset_name[ds_indice]} Repetition {rep}: {best_layerwise_acc[dataset_name[ds_indice]][rep]}")

In [ ]:
# Fine Tuned

fine_tuned_acc = {}

for k in range(0, len(dataset_name)):
    refined_acc = []
    df = pd.read_json(f"./{model_name}/Data/Core/{refining_type}/Refined_Accuracy/Fine_Tuned/{dataset_name[k]}_Accuracy.json")

    for i in range(5):
        refined_acc.append(df["Accuracy"][i])
    fine_tuned_acc[dataset_name[k]] = refined_acc

    print(f"{model_name} - {dataset_name[k]}: Fine Tuned Accuracy")
    ci_low, ci_high = scipy.stats.t.interval(0.95, df=len(refined_acc)-1, loc=np.mean(refined_acc), scale=sem(refined_acc))
    print(f"Fine-Tuned Average: {np.mean(refined_acc)}, Error: +- {ci_high-np.mean(refined_acc)}, 95% Confidence Interval: {(ci_low, ci_high)}\n")

In [ ]:
# Linear Probe

linear_probe_acc = {}

for k in range(0, len(dataset_name)):
    refined_acc = []
    df = pd.read_json(f"./{model_name}/Data/Core/{refining_type}/Refined_Accuracy/Linear_Probe/{dataset_name[k]}_Accuracy.json")

    for i in range(5):
        refined_acc.append(df["Accuracy"][i])
    linear_probe_acc[dataset_name[k]] = refined_acc
    
    print(f"{model_name} - {dataset_name[k]}: Linear Probe Accuracy")
    ci_low, ci_high = scipy.stats.t.interval(0.95, df=len(refined_acc)-1, loc=np.mean(refined_acc), scale=sem(refined_acc))
    print(f"Fine-Tuned Average: {np.mean(refined_acc)}, Error: +- {ci_high-np.mean(refined_acc)}, 95% Confidence Interval: {(ci_low, ci_high)}\n")

In [ ]:
# Ablation

best_layerwise_f_t_Classifier = {}
best_layerwise_f_t_Classifier_index = {}


def find_best_acc(data):
    acc = {i: [] for i in indices} # Indices, then len(data)

    for i in indices:
        for j in range(len(data)):
            acc[i].append(data[j]["Classification_Accuracy"][i])

    best_acc = {}

    for i in acc:
        arr = acc[i]
        max_val = max(arr)
        best_acc[i] = max_val

    final = []
    for i in indices:
        final.append(best_acc[i])

    best_accuracy = max(final)
    index = final.index(best_accuracy)

    return best_accuracy, index

# Ablation Base with Fine Tuned Classifier Head
# big_data = {i: {} for i in range(len(dataset_name))}

for k in range(len(dataset_name)):
    best_layerwise_f_t_Classifier[dataset_name[k]] = {}
    best_layerwise_f_t_Classifier_index[dataset_name[k]] = {}
    said=k
    results_path = f"./{model_name}/Data/Core/{refining_type}/{refiner}/{dataset_name[k]}/{domain}" # /Entire_Transformation_Matrix_W"

    big_data = {}
    for i in range(1,6):
        results[i] = []
        path = f"{results_path}/{i}/Entire_Transformation_Matrix_W"
        try:
            for filename in os.listdir(path):
                if filename in [f"Base_Fine_Tuned_Classifier_Results_{i}.json"]: 
                    file_path = os.path.join(path, filename)
                    if os.path.isfile(file_path):
                        results[i].append(file_path)
        except FileNotFoundError:
            print(f"Error: The Folder '{path}' was not found.")
        except Exception as e:
            print(f"An error occured: {e}")

        data = [pd.read_json(i) for i in results[i]]
        results[i] = data
        big_data[i] = results[i]
    
    best_acc = []
    index = []
    print(f"{model_name} - {dataset_name[said]}: Ablation Base with Fine-Tuned Classifier")
    for i in range(1,6):
        acc, ind = find_best_acc(big_data[i])
        best_acc.append(acc)
        index.append(ind)
        print(f"Set {i} Best Accuracy: {best_acc[i-1]} | Transformation Layer (0-11): {index[i-1]}")
        best_layerwise_f_t_Classifier[dataset_name[k]][i] = retrieve_val(f"{results_path}/{i}/Entire_Transformation_Matrix_W/Base_Fine_Tuned_Classifier_Results_{i}.json")
        best_layerwise_f_t_Classifier_index[dataset_name[k]][i] = ind
    
    ci_low, ci_high = scipy.stats.t.interval(0.95, df=len(best_acc)-1, loc=np.mean(best_acc), scale=sem(best_acc))
    print(f"Average: {np.mean(best_acc)}, Error: +- {ci_high-np.mean(best_acc)}, 95% Confidence Interval: {(ci_low, ci_high)}\n")

In [ ]:
for ds_indice in range(len(dataset_name)):
    for rep in range(1,6):
        print(best_layerwise_f_t_Classifier[dataset_name[ds_indice]][rep])

### Graphing Data

In [ ]:
# best_layerwise_acc = {}
# best_layerwise_img = {}
# best_layerwise_index = {}
# fine_tuned_acc = {}
# linear_probe_acc = {}
# best_layerwise_f_t_Classifier = {}
# best_layerwise_f_t_Classifier_index = {}

In [ ]:
task_matrix_indice_mean_acc = {}
ablation_indice_mean_acc = {}

In [ ]:
for ds_indice in range(len(dataset_name)):
    task_matrix_indice_mean_acc[dataset_name[ds_indice]] = []
    ablation_indice_mean_acc[dataset_name[ds_indice]] = []
    for idx in indices:
        temp_acc = []
        temp_ablation_acc = []
        for rep in range(1,6):
            temp_acc.append(best_layerwise_acc[dataset_name[ds_indice]][rep][idx])
            temp_ablation_acc.append(best_layerwise_f_t_Classifier[dataset_name[ds_indice]][rep][idx])
        task_matrix_indice_mean_acc[dataset_name[ds_indice]].append(np.mean(temp_acc))
        ablation_indice_mean_acc[dataset_name[ds_indice]].append(np.mean(temp_ablation_acc))

In [ ]:
for ds_indice in range(len(dataset_name)):
    # print(task_matrix_indice_mean_acc[dataset_name[ds_indice]])
    print(ablation_indice_mean_acc[dataset_name[ds_indice]])

In [ ]:
layerwise_results_path = f"./{model_name}/Graphs/{refining_type}/{refiner}"
os.makedirs(layerwise_results_path, exist_ok=True)

In [ ]:
for ds_indice in range(len(dataset_name)):
    plt.plot(indices, task_matrix_indice_mean_acc[dataset_name[ds_indice]], marker = "o", label="Task Matrix")
    plt.plot(indices, ablation_indice_mean_acc[dataset_name[ds_indice]], marker = "o", label="Base w/Fine-Tuned Classifier")
    plt.plot(indices, [np.mean(fine_tuned_acc[dataset_name[ds_indice]])]*12, linestyle="--", label="Fine-Tuned")
    plt.plot(indices, [np.mean(linear_probe_acc[dataset_name[ds_indice]])]*12, linestyle="--", label="Linear Probe")
    plt.legend(
        loc="center left",
        bbox_to_anchor=(1.0, 0.5)
    )
    plt.title(f"CLIP ViT Vision {dataset_name[ds_indice]} Best Layerwise Results")
    plt.xlabel("Layer")
    plt.ylabel("Classification Accuracy")
    plt.savefig(f"{layerwise_results_path}/{dataset_name[ds_indice]}_Layerwise_Accuracy", dpi=600, bbox_inches='tight')
    plt.show()

## Scaling Coefficient